# ЛР №2. Регрессия: построение первой модели машинного обучения

**Задача:** предсказать стоимость квартиры (`PriceMillion`) по её характеристикам.

В этой работе мы впервые проходим полный ML-pipeline: признаки → train/test → baseline → модель → прогноз → метрики → анализ ошибок.


In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# from sklearn.model_selection import train_test_split
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.pipeline import Pipeline
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("ai_housing_lab2.csv")
df.head()


,Area,Rooms,Floor,BuildingAge,MetroMinutes,District,Renovation,PriceMillion
0,48.9,1,22,36,10.4,Центр,Обычный,14.40
1,102.9,3,8,13,5.9,Центр,Обычный,30.35
2,82.7,3,2,43,21.3,Центр,Без ремонта,17.19
3,71.3,2,17,51,10.3,Юг,Хороший,17.58
4,69.5,3,23,34,12.0,Юг,Обычный,15.59


## Задание 1. Постановка задачи
Определите объект наблюдения, целевую переменную и признаки. Объясните, почему это задача **регрессии**, а не классификации.


# Объект набюдения квартиры с фиксированной стоимостью.

Целевая переменная: стоимость квартиры

Потому что целевая переменная это числовое значение, а не категория

In [18]:
# Ваш код для первичного просмотра

print(df.shape)
print(df.columns)
print(df.dtypes) 



(650, 8)
Index(['Area', 'Rooms', 'Floor', 'BuildingAge', 'MetroMinutes', 'District',
       'Renovation', 'PriceMillion'],
      dtype='str')
Area            float64
Rooms             int64
Floor             int64
BuildingAge       int64
MetroMinutes    float64
District            str
Renovation          str
PriceMillion    float64
dtype: object


## Задание 2. X и y
Отделите `PriceMillion` от признаков. Определите числовые и категориальные столбцы.


In [19]:
X = df.drop(columns="PriceMillion")
y = df["PriceMillion"]

cat_cols = ["District", "Renovation"]
num_cols = [
"Area", "Rooms", "Floor",
"BuildingAge", "MetroMinutes"
]

## Задание 3. Train/test split
Разделите данные в отношении 80/20 с `random_state=42`. Покажите размеры выборок. Объясните, зачем тестовые данные не должны участвовать в обучении.


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
X, y,
test_size=0.2,
random_state=42
)

## Задание 4. Baseline
Создайте простейший прогноз: для каждого объекта предсказывайте **среднюю цену обучающей выборки**. Рассчитайте MAE и RMSE. Эти значения станут точкой отсчёта.


In [21]:
from sklearn.metrics import (
mean_absolute_error,
mean_squared_error,
r2_score
)

baseline_pred = np.full(
len(y_test),
y_train.mean()
)

baseline_mae = mean_absolute_error(
y_test, baseline_pred
)
baseline_rmse = mean_squared_error(
y_test, baseline_pred
) ** 0.5

baseline_r2 = r2_score(
y_test, baseline_pred
)
print(baseline_mae, baseline_rmse, baseline_r2)

6.555439349112427 7.652992047072777 -0.005648562396039392


## Задание 5. Linear Regression
Создайте preprocessing через `ColumnTransformer`: категориальные признаки кодируйте `OneHotEncoder(handle_unknown='ignore')`, числовые передавайте без изменения. Объедините preprocessing и `LinearRegression()` в `Pipeline`, обучите модель.


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# 1. Создаем препроцессинг: 
# - для категориальных столбцов применяем OneHotEncoder
# - для числовых столбцов используем 'passthrough' (оставляем как есть)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# 2. Создаем Pipeline, который сначала применяет preprocessor, а затем LinearRegression
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 3. Обучаем весь пайплайн на тренировочных данных
model_pipeline.fit(X_train, y_train)

print("Модель успешно обучена!")

Модель успешно обучена!


## Задание 6. Метрики
Получите прогноз на test и вычислите:
- MAE;
- RMSE;
- R².

Сравните MAE/RMSE с baseline. Объясните метрики своими словами и укажите единицы измерения MAE/RMSE.


In [23]:
y_pred = model_pipeline.predict(X_test)
mae_pred = mean_absolute_error(y_test, y_pred)
mse_pred = mean_squared_error(y_test, y_pred)
rmse_pred = mse_pred ** 0.5
r2_pred = r2_score(y_test, y_pred)

print(f"{y_pred=}, {mae_pred=}, {mse_pred=}, {rmse_pred=}, {r2_pred=},")

y_pred=array([21.82026085, 20.45634606, 27.9903428 , 12.02490252, 27.63595304,
       17.51694572, 23.84602097, 26.79780945, 11.20529017,  7.67389596,
       12.18238812, 10.9516113 , 13.55827805, 25.38797399, 19.68939472,
       10.93734488, 23.44997863,  7.76734565, 16.40648448, 26.30572648,
       29.5976719 , 11.0969155 , 18.55278194, 17.23661548, 24.17725361,
       24.36150469, 12.35137967, 22.56106745, 13.72391532, 25.79304552,
       23.42196384, 24.85440037, 30.46542631, 27.44085221, 29.50864435,
       26.2251392 , 22.89493521, 18.7000449 , 12.06855121, 15.85791716,
        8.06678906, 27.43048023, 21.0660156 , 10.98937281, 25.76137778,
       18.18704291, 16.93598282, 30.85997853, 34.75495652, 23.3464863 ,
       18.48066389, 30.16554446, 18.47119172, 12.06896779, 18.4918241 ,
       33.07412901, 30.4111836 ,  8.34438731, 12.0332236 , 24.25451411,
        9.38204043, 15.70103813, 11.8481872 , 14.29976961, 26.50430071,
       34.08070776, 13.06291531, 19.4169206 , 28.74795484

In [24]:
metrics_df = pd.DataFrame({
    'Метрика': ['MAE', 'RMSE', 'R²'],
    'Baseline': [baseline_mae, baseline_rmse, baseline_r2], 
    'Наша модель': [mae_pred, rmse_pred, r2_pred]
})

print(metrics_df)

  Метрика  Baseline  Наша модель
0     MAE  6.555439     1.840226
1    RMSE  7.652992     2.329113
2      R² -0.005649     0.906854


## Задание 7. Факт и прогноз
Постройте scatter plot: по X — реальные цены, по Y — прогноз. Добавьте диагональ идеального прогноза `y=x`. Чем ближе точки к диагонали, тем точнее модель.


In [25]:
# Ваш код

## Задание 8. Ошибки модели
Создайте таблицу `Actual`, `Predicted`, `Error`, `AbsError`. Найдите 10 объектов с наибольшей абсолютной ошибкой. Есть ли у них общие особенности?


In [26]:
# Ваш код

## Задание 9. Эксперимент с признаками
Обучите минимум три версии:
1. только `Area`;
2. только числовые признаки;
3. все признаки.

Для каждой запишите MAE, RMSE, R². Сделайте вывод: всегда ли добавление признаков улучшает модель?


In [27]:
# Ваш код

## Задание 10. Прогноз для новых объектов
Создайте DataFrame минимум из трёх вымышленных квартир и получите прогноз цены. Один объект сделайте близким к типичным данным, другой — дорогим, третий — необычным/пограничным. Объясните, почему прогноз за пределами диапазона обучающих данных следует трактовать осторожно.


In [28]:
# Ваш код

## Итог
Ответьте:
1. Насколько модель лучше baseline?
2. Какая метрика наиболее понятна в единицах задачи?
3. Что означает R²?
4. Какие объекты модель предсказывает хуже?
5. Почему хорошая метрика на test ещё не доказывает, что модель готова к реальной эксплуатации?
